# Aim Objectives and Outline

**Researcher: Mercy Mamkon Oyeleke**  
**AIR QUALITY ANALYSIS RESEARCH ANALYSIS**

## Research Background and Goals

This research studies the relationship between weather and ground-level ozone. It focuses on Ontario so that the dashboard is relevant to communities across the province. Comparing selected locations makes local differences visible rather than using one result for the whole province.

Ozone was selected because its formation and movement are linked to weather. Temperature, wind speed and sunlight provide a clear starting point for investigating these patterns. These data are available through documented sources and can be updated. They do not describe all the factors that affect ozone.

This research demonstrates my ability to prepare data, explain analytical choices, check predictions and present useful findings. Its purpose is to support understanding and planning, while making uncertainty clear.

## Research Aim

This research examines how temperature, wind speed and sunlight relate to ground-level ozone, and whether these relationships can help estimate afternoon ozone in selected Ontario locations.

## Research Objectives

1. **Understand the data:** describe its sources, dates, units, coverage and missing values so the analysis is based on clearly defined information.
2. **Choose a data preparation approach:** compare complete cases, mean filling and median filling in the historical demonstration, and explain their limitations. Keep filled ozone responses out of modern prediction checks.
3. **Explore relationships:** use charts and Pearson and Spearman correlations to understand how weather and ozone vary together, without assuming cause and effect.
4. **Compare methods:** examine temperature-only, multiple regression and log-transformed models in the historical analysis. Explain why an easier-to-interpret model is a useful starting point, without declaring a winner before suitable testing.
5. **Check predictions:** evaluate the Ontario model on later days kept out of training, report errors in ozone units and show uncertainty. Keep training results separate from results on unseen days.
6. **Explain practical uses:** present findings in a dashboard and describe possible uses for public health, healthcare, schools, outdoor employers and environmental teams, including what extra evidence each use needs.

## Research Questions

- How do temperature, wind and sunlight relate to afternoon ozone?
- How do missing-data choices change the historical analysis?
- Does using several weather inputs describe the historical data better than temperature alone?
- How closely does the Ontario model estimate source ozone values on later days kept out of training?
- How could the information support planning, and where are its limits?

## Research Workflow Pipeline

![Process flow showing separate historical analysis and Ontario data collection, preparation, prediction checks, forecasting and dashboard steps](../docs/workflow_pipeline.png)

The image shows how this research moves from source data to estimates and planning information. The historical method is separate from the Ontario prediction model.

## Research Outline

| Section | What it explains | Why it matters |
|---|---|---|
| 1. Summary | The question, approach and main findings | Gives the reader a clear starting point |
| 2. Background, aim and objectives | Why this research was undertaken and what it investigates | Keeps the work focused |
| 3. Data sources and understanding | Locations, dates, units and missing values | Shows what the evidence represents |
| 4. Data preparation | Missing-data choices and time alignment | Makes analytical decisions transparent |
| 5. Charts and relationships | Distributions and correlations | Reveals patterns and possible problems |
| 6. Historical model comparison | Simple, multiple and log-transformed regression | Explains the methods using the 1973 data |
| 7. Ontario prediction checks | Later-day evaluation, errors and uncertainty | Separates model fitting from prediction evidence |
| 8. Dashboard and industry applications | How readers explore results and possible uses | Connects analysis with practical questions |
| 9. Limitations and next steps | What the research cannot establish and what to test next | Prevents conclusions that go beyond the evidence |

## Boundaries Of The Research

The 1973 New York dataset is a separate method demonstration, not the source of today's Ontario predictions. The Ontario model uses recent data for each selected location. Modern training ozone comes from CAMS through Open-Meteo and represents model estimates, not independent station measurements.

The historical raw and log models may be compared using errors on the same original ozone scale, but errors calculated on training rows do not prove future forecasting performance. Raw-response and log-response R-squared, AIC and BIC cannot simply be ranked as if they describe the same response scale. Non-significant diagnostic tests do not prove that every model assumption is satisfied.

Complete-case analysis avoids inserting invented response values but can still introduce bias if the missing rows differ systematically. Low VIF values reduce concern about overlapping predictors; they do not prove that every source of model instability is absent.

The research does not establish cause and effect, predict hospital visits, validate health alerts or demonstrate compliance with official air-quality standards. Its afternoon ozone tiers are illustrative and differ from official AQHI and eight-hour regulatory measures. Independent station validation and testing across seasons are important next steps.

## Corrections Made After Reviewing The Draft

The reviewed notebook reruns the analysis from the saved canonical data instead of copying the draft's headline figures. Its raw multiple-regression training RMSE is about 20.80 ppb, and the exponentiated log-model training RMSE is about 19.53 ppb. The draft's 17.91 ppb result was not reproduced. These figures describe fit to training rows, not future forecasting accuracy.

The rerun also does not support the draft's claim that the log model resolves both residual normality and unequal error variance. The notebook reports the calculated diagnostic results directly. No historical model is labelled a validated forecasting winner.


# Research Summary

This research explores how weather relates to afternoon ozone. The historical notebook explains the methods; the Ontario dashboard uses separate recent location models. The results support further investigation rather than validated health alerts.

Historical complete cases: 111. Raw multiple-regression training RMSE: 20.80 ppb. Log-model training RMSE after exponentiation: 19.53 ppb. These are training errors, not unseen-day forecasting results.

The rerun does not reproduce the supplied draft's 17.91 ppb log-model error or its claim that the log model resolves both diagnostic problems. The diagnostic table below contains the actual results from this run; a model winner is not declared from those draft claims.


# Historical Ozone Analysis
1973 NYC data, used to demonstrate method only.
Run from the repository root or notebooks directory. Source: canonical R airquality via statsmodels / saved CSV.
Complete cases are primary; missing-data approaches are descriptive sensitivity analyses.
Log-response statistics use log(ppb), so do not rank its AIC/BIC or RMSE against raw-response models.


## 1. Data Sources and Missing Values


In [1]:
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd

from src.api_clients import baseline
from src.modeling import baseline_frame, fit, metrics, missing_comparison, normality, vif

raw = baseline_frame(baseline(True))
clean = raw.dropna()
print(missing_comparison(raw).to_string(index=False))

       Method   n  Ozone SD  Pearson r with Temp
Complete case 111 33.275969             0.698541
         Mean 153 28.693372             0.608742
       Median 153 29.054449             0.600833


## 2. Missing-data Choices


In [2]:
from src.modeling import baseline_sensitivity

sensitivity = baseline_sensitivity(raw)
print(sensitivity.to_string(index=False))
print('Imputed responses are descriptive sensitivity analysis only, never live validation targets.')
print('Training RÂ² differs across changed samples; this is not evidence of comparative predictive skill.')

       Method   n       R²  Adjusted R²         AIC         BIC      RMSE  Residual Shapiro p  Imputed ozone targets  Intercept  Temperature coefficient (ppb/°F)  Wind coefficient (ppb/mph)  Solar coefficient (ppb/Langley)
Complete case 111 0.605895     0.594845  996.717103 1007.555224 20.795614        3.617786e-06                      0 -64.342079                          1.652093                   -3.333591                         0.059821
         Mean 153 0.480025     0.469556 1368.273503 1380.395255 20.622851        7.827876e-08                     37 -38.223155                          1.241263                   -2.717250                         0.057747
       Median 153 0.472094     0.461465 1374.416454 1386.538206 21.041038        7.603676e-08                     37 -39.354064                          1.232953                   -2.787086                         0.056958
Imputed responses are descriptive sensitivity analysis only, never live validation targets.
Training RÂ² dif

## 3. Distributions and Weather Relationships


In [3]:
print(normality(clean).to_string(index=False))
print('Pearson correlation')
print(clean.corr().round(3).to_string())
print('Spearman correlation')
print(clean.corr(method='spearman').round(3).to_string())

Variable        W            p
   ozone 0.873554 2.846396e-08
    temp 0.980067 9.569113e-02
    wind 0.980764 1.098834e-01
   solar 0.932850 2.957179e-05
Pearson correlation
       ozone   temp   wind  solar
ozone  1.000  0.699 -0.612  0.348
temp   0.699  1.000 -0.497  0.294
wind  -0.612 -0.497  1.000 -0.127
solar  0.348  0.294 -0.127  1.000
Spearman correlation
       ozone   temp   wind  solar
ozone  1.000  0.773 -0.605  0.348
temp   0.773  1.000 -0.499  0.210
wind  -0.605 -0.499  1.000 -0.062
solar  0.348  0.210 -0.062  1.000


## 4. Historical Model Comparison


In [4]:
simple = fit(clean, ['temp'])
multiple = fit(clean)
log_model = fit(clean, log=True)
comparison = pd.DataFrame([metrics(m) for m in [simple, multiple, log_model]], index=['Simple', 'Multiple', 'Log response'])
print(comparison.to_string())
print(pd.DataFrame({'coefficient': multiple.params, 'p': multiple.pvalues}).to_string())
print(vif(clean).to_string(index=False))

                n        R²  Adjusted R²          AIC          BIC       RMSE  Residual Shapiro p
Simple        111  0.487960     0.483262  1021.775145  1027.194205  23.703772        1.522253e-07
Multiple      111  0.605895     0.594845   996.717103  1007.555224  20.795614        3.617786e-06
Log response  111  0.664424     0.655015   168.825058   179.663178   0.499323        5.726116e-02
       coefficient             p
const   -64.342079  6.226638e-03
temp      1.652093  2.423506e-09
wind     -3.333591  1.515934e-06
solar     0.059821  1.123664e-02
Feature      VIF
   temp 1.431367
   wind 1.329070
  solar 1.095253


## 5. Analytical Charts


In [5]:
from src import viz

# Interactive figures display when re-running in Jupyter.
figures = [viz.histogram(clean.ozone, 'Ozone (ppb)'), viz.qq(clean.ozone, 'Ozone (ppb)'), viz.heatmap(clean, 'pearson'), viz.residual(multiple)]
for figure in figures:
    print(figure.layout.title.text)
# Uncomment to view in an interactive notebook:
# for figure in figures: figure.show()

Distribution · ozone
Normal Q–Q · ozone
Pearson correlation
Residual diagnostics


## 6. Historical Errors and Diagnostic Checks


In [6]:
import numpy as np
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera

rows = []
for name, model in [('Raw multiple regression', multiple), ('Log response, exp prediction', log_model)]:
    fitted = np.exp(model.fittedvalues) if model is log_model else model.fittedvalues
    rows.append({'Model': name, 'Training RMSE (ppb)': float(np.sqrt(np.mean((clean.ozone-fitted)**2))), 'Breusch-Pagan p': het_breuschpagan(model.resid, model.model.exog)[1], 'Jarque-Bera p': jarque_bera(model.resid)[1], 'Durbin-Watson': durbin_watson(model.resid)})
print(pd.DataFrame(rows).to_string(index=False))
print('These errors use training rows. They do not establish future forecast accuracy.')
print('Exp(log prediction) estimates the conditional median under the log model assumptions, not a bias-corrected mean.')
print('A non-significant diagnostic test does not prove that all assumptions hold.')


                       Model  Training RMSE (ppb)  Breusch-Pagan p  Jarque-Bera p  Durbin-Watson
     Raw multiple regression            20.795614         0.167788   3.852421e-19       1.935476
Log response, exp prediction            19.528119         0.000339   5.639409e-04       1.806760
These errors use training rows. They do not establish future forecast accuracy.
Exp(log prediction) estimates the conditional median under the log model assumptions, not a bias-corrected mean.
A non-significant diagnostic test does not prove that all assumptions hold.


## 7. Ontario Application and Practical Uses
The calculations above describe historical New York data only. The Ontario dashboard fits separate recent location models and checks them on later days kept out of training.
Ontario source ozone is modelled CAMS data, not independent station readings. See ../docs/plain_language_guide.md for industry examples and ../docs/research_outline.md for objectives and boundaries.
The native Power BI report is still awaiting a successful display check; no verified PBIX is claimed.
